# 🚀 OpenSource Clipping — Quick Start Notebook

> **⚡ Quick Start**: If you just want to get up and running as fast as possible,
> simply run all cells from top to bottom. The only thing you need to change is
> the **YouTube URL** and your **API Key** in the Configuration section below.

---

## What is this?

This notebook is a **minimal, beginner-friendly** implementation of
[OpenSource Clipping](https://github.com/NaufalRizqullah/opensource-clipping) —
an AI-powered tool that automatically transforms long-form videos into
cinematic short-form highlight clips with:

- 🎯 AI-curated highlight moments (powered by Google Gemini)
- 🎙️ Word-level transcription (powered by Faster-Whisper)
- 📱 Smart face-tracking auto-framing for vertical (9:16) format
- 💬 Karaoke-style animated subtitles
- 🎬 Cinematic hook teaser intro with glitch transition

## How to use this notebook

| Step | Section | What to do |
|------|---------|------------|
| 1 | **Setup** | Run the setup cells to clone the repo & install dependencies |
| 2 | **Configuration** | Set your YouTube URL, API key, and clip preferences |
| 3 | **Run** | Execute the pipeline — sit back and wait for your clips! |
| 4 | **Download** | Download the generated clips from the `outputs/` folder |

### ⚠️ Requirements

- **Runtime**: Set your Colab runtime to **T4 GPU** (`Runtime > Change runtime type > T4 GPU`)
- **API Key**: You need a free [Google Gemini API Key](https://aistudio.google.com/apikey)

---

## 1. Setup — Clone Repository & Install Dependencies

This section clones the OpenSource Clipping repository and installs all
required Python packages. It also installs **FFmpeg** (needed for video
processing) and **Deno** (used internally by some modules).

> 💡 **Tip**: This cell only needs to run once per Colab session.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# 1a. Clone the repository into the current working directory
# ──────────────────────────────────────────────────────────────────────
!rm -rf ./* ./.*
!git clone https://github.com/NaufalRizqullah/opensource-clipping.git .

In [ ]:
%%capture
# ──────────────────────────────────────────────────────────────────────
# 1b. Install system dependencies (FFmpeg + Deno) and Python packages
# ──────────────────────────────────────────────────────────────────────
import os

# FFmpeg — required for video encoding/decoding
!apt-get -qq update
!apt-get -qq install -y ffmpeg

# Deno — used by internal subtitle rendering modules
!curl -fsSL https://deno.land/install.sh | sh
os.environ["PATH"] += ":/root/.deno/bin"

# Python dependencies
!pip install -r requirements.txt

## 2. Configuration — Set Your Preferences

This is the **only section you need to edit**. Configure the following:

| Parameter | Description | Default |
|-----------|-------------|---------|
| `YOUTUBE_URL` | The YouTube video URL you want to clip | *(required)* |
| `TOTAL_CLIPS` | How many highlight clips to generate | `3` |
| `WHISPER_MODEL` | Whisper model size for transcription accuracy | `large-v3` |
| `WHISPER_DEVICE` | Device for Whisper inference (`cuda` or `cpu`) | `cuda` |
| `WHISPER_COMPUTE_TYPE` | Compute precision (`float16` for Colab T4, `float32` for CPU) | `float16` |

### Whisper Model Options

| Model | Speed | Accuracy | VRAM |
|-------|-------|----------|------|
| `tiny` | ⚡⚡⚡⚡⚡ | ★☆☆☆☆ | ~1 GB |
| `base` | ⚡⚡⚡⚡ | ★★☆☆☆ | ~1 GB |
| `small` | ⚡⚡⚡ | ★★★☆☆ | ~2 GB |
| `medium` | ⚡⚡ | ★★★★☆ | ~5 GB |
| `large-v3` | ⚡ | ★★★★★ | ~10 GB |

> 💡 **Recommendation**: Use `large-v3` with `float16` on Colab T4 GPU for best
> results. If you're on a free Kaggle notebook or CPU-only environment, use
> `small` or `medium` with `float32`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CONFIGURATION — Edit the values below to match your preferences
# ══════════════════════════════════════════════════════════════════════

# ── YouTube URL ──────────────────────────────────────────────────────
# Paste the YouTube video URL you want to generate clips from.
YOUTUBE_URL = "https://www.youtube.com/watch?v=YOUR_VIDEO_ID"  # <-- CHANGE THIS

# ── Total Clips ─────────────────────────────────────────────────────
# Number of highlight clips to generate from the video.
# More clips = longer processing time. Start small (1-3) for testing.
TOTAL_CLIPS = 3

# ── Whisper Configuration ────────────────────────────────────────────
# These control the speech-to-text transcription step.
WHISPER_MODEL = "large-v3"        # Model size (see table above)
WHISPER_DEVICE = "cuda"            # "cuda" for GPU, "cpu" for CPU-only
WHISPER_COMPUTE_TYPE = "float16"   # "float16" for Colab T4, "float32" for CPU/Kaggle

# ══════════════════════════════════════════════════════════════════════
#  OPTIONAL — Advanced settings (safe to leave as defaults)
# ══════════════════════════════════════════════════════════════════════

ASPECT_RATIO = "9:16"              # Output aspect ratio: "9:16", "16:9", "1:1"
FONT_STYLE = "HORMOZI"             # Subtitle style: "DEFAULT", "HORMOZI", "CINEMATIC", "STORYTELLER"
HOOK_DURATION = 3                  # Hook teaser duration in seconds
WORDS_PER_SUBTITLE = 5             # Max words per subtitle group

print("✅ Configuration loaded!")
print(f"   URL           : {YOUTUBE_URL}")
print(f"   Total Clips   : {TOTAL_CLIPS}")
print(f"   Whisper Model : {WHISPER_MODEL} ({WHISPER_DEVICE}, {WHISPER_COMPUTE_TYPE})")
print(f"   Aspect Ratio  : {ASPECT_RATIO}")
print(f"   Font Style    : {FONT_STYLE}")

## 3. API Key Setup

Your **Google Gemini API Key** is required for the AI analysis step.
The recommended way is to store it in **Colab Secrets**:

1. Click the 🔑 **Key icon** in the left sidebar of Colab
2. Add a new secret named `GOOGLE_API_KEY`
3. Paste your API key as the value
4. Toggle the **"Notebook access"** switch ON

> 🔗 Get a free API key at: https://aistudio.google.com/apikey

**Optional keys** (for extra features):
- `PEXELS_API_KEY` — Enables automatic B-roll stock footage ([get one here](https://www.pexels.com/api/))
- `HF_TOKEN` — Enables split-screen / camera-switch mode ([get one here](https://huggingface.co/settings/tokens))

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Load API keys from Colab Secrets and create the .env file
# ──────────────────────────────────────────────────────────────────────
from google.colab import userdata
from pathlib import Path

# Required
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY") or ""

# Optional — leave blank if you don't have them
PEXELS_API_KEY = userdata.get("PEXELS_API_KEY") or ""
HF_TOKEN = userdata.get("HF_TOKEN") or ""

# Write the .env file so the pipeline can read the keys
env_text = f"""# Auto-generated by Quick Start notebook
GOOGLE_API_KEY={GOOGLE_API_KEY}
PEXELS_API_KEY={PEXELS_API_KEY}
HF_TOKEN={HF_TOKEN}
"""
Path(".env").write_text(env_text, encoding="utf-8")

# Status check
print("🔑 API Key Status:")
print(f"   GOOGLE_API_KEY : {'✅ Set' if GOOGLE_API_KEY else '❌ Missing (required!)'}")
print(f"   PEXELS_API_KEY : {'✅ Set' if PEXELS_API_KEY else '⚪ Not set (optional)'}")
print(f"   HF_TOKEN       : {'✅ Set' if HF_TOKEN else '⚪ Not set (optional)'}")

if not GOOGLE_API_KEY:
    print("\n⚠️  WARNING: GOOGLE_API_KEY is missing!")
    print("   Add it to Colab Secrets (🔑 icon in sidebar) before running the pipeline.")
    print("   Get your free key at: https://aistudio.google.com/apikey")

## 4. Run the Pipeline 🎬

This cell runs the full AI clipping pipeline:

1. **Download** — Downloads the YouTube video
2. **Transcribe** — Converts speech to text using Faster-Whisper
3. **Analyze** — Google Gemini AI picks the most engaging moments
4. **Render** — Generates clips with face-tracking, subtitles, and hook teasers

> ⏱️ **Estimated time**: 5–15 minutes depending on video length and number of clips.

> ⚠️ Make sure you have set the `YOUTUBE_URL` in the Configuration section above!

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# YouTube Cookies Setup (Optional) - Solves yt-dlp 403 Forbidden errors
# ──────────────────────────────────────────────────────────────────────
import os
import shutil

COOKIES_PATH = os.path.abspath("youtube_cookies.txt")
cookie_candidates = [
    "/kaggle/input/datasets/muhammadnaufal/tiktok-secret-cookies/youtube_cookies.txt",
    "/content/youtube_cookies.txt",
    COOKIES_PATH,
]

source_cookie = next((p for p in cookie_candidates if os.path.isfile(p)), None)
YT_COOKIES_ARG = ""

if source_cookie:
    if os.path.realpath(source_cookie) != os.path.realpath(COOKIES_PATH):
        shutil.copy(source_cookie, COOKIES_PATH)
    print(f"✅ YouTube cookies aktif: {COOKIES_PATH}")
    YT_COOKIES_ARG = f"--yt-cookies {COOKIES_PATH}"
else:
    print("ℹ️ youtube_cookies.txt tidak ditemukan. Download akan dicoba tanpa cookies.")


In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Execute the OpenSource Clipping pipeline with your configuration
# ──────────────────────────────────────────────────────────────────────

!python main.py \
  --url "{YOUTUBE_URL}" \
  --clips {TOTAL_CLIPS} \
  --ratio "{ASPECT_RATIO}" \
  --font-style "{FONT_STYLE}" \
  --hook-duration {HOOK_DURATION} \
  --words-per-sub {WORDS_PER_SUBTITLE} \
  --whisper-model "{WHISPER_MODEL}" \
  --whisper-device "{WHISPER_DEVICE}" \
  --whisper-compute-type "{WHISPER_COMPUTE_TYPE}" \
  {YT_COOKIES_ARG}

## 5. View & Download Results

After the pipeline completes, your clips are saved in the `outputs/` folder.
Run the cells below to preview and download them.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# List all generated output files
# ──────────────────────────────────────────────────────────────────────
import os

outputs_dir = "outputs"
if os.path.exists(outputs_dir):
    files = os.listdir(outputs_dir)
    video_files = [f for f in files if f.endswith(".mp4")]
    print(f"🎬 Generated {len(video_files)} clip(s):\n")
    for f in sorted(files):
        size_mb = os.path.getsize(os.path.join(outputs_dir, f)) / (1024 * 1024)
        icon = "🎥" if f.endswith(".mp4") else "🖼️" if f.endswith((".jpg", ".png")) else "📄"
        print(f"   {icon} {f} ({size_mb:.1f} MB)")
else:
    print("❌ No outputs found. Make sure the pipeline ran successfully.")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Download all output files as a ZIP archive
# ──────────────────────────────────────────────────────────────────────
import shutil
from google.colab import files

if os.path.exists(outputs_dir) and os.listdir(outputs_dir):
    zip_name = "clipping_outputs"
    shutil.make_archive(zip_name, "zip", outputs_dir)
    print(f"📦 Created {zip_name}.zip — downloading now...")
    files.download(f"{zip_name}.zip")
else:
    print("❌ No outputs to download.")

---

## 📖 Troubleshooting

| Problem | Solution |
|---------|----------|
| `GOOGLE_API_KEY not found` | Add your API key to Colab Secrets (🔑 sidebar) |
| `CUDA out of memory` | Use a smaller Whisper model (`small` or `medium`) |
| `float16 not supported` | Change `WHISPER_COMPUTE_TYPE` to `float32` |
| Pipeline takes too long | Reduce `TOTAL_CLIPS` to `1` for testing |
| No B-roll footage | Add `PEXELS_API_KEY` to Colab Secrets |

## 🔗 Links

- **GitHub**: [NaufalRizqullah/opensource-clipping](https://github.com/NaufalRizqullah/opensource-clipping)
- **Full Documentation**: See the [README](https://github.com/NaufalRizqullah/opensource-clipping/blob/main/README.md) for all CLI options
- **Get Gemini API Key**: [aistudio.google.com/apikey](https://aistudio.google.com/apikey)
- **Get Pexels API Key**: [pexels.com/api](https://www.pexels.com/api/)